# 04 — Pooled classifier and next-bar regressor

**Research notebook only** — Streamlit remains a results/deployment interface. Uses the persisted **220-stock master table** from notebook 03, with the exact same training-fitted standardization, one-hot encoding, candle clusters, quantile thresholds, and feature order. No PostgreSQL writes.

**Important:** this is an initial chronological holdout. The holdout must not be repeatedly used to choose features or models; walk-forward validation and a separate untouched final test set are next.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from stock_ml_dynamic.pooled_models import (load_master, prepare_pooled_data,
    classification_experiment, regression_experiment, predict_latest, save_pooled_run)

INTERVAL = '1d'
MASTER_DIRECTORY = Path('artifacts/master_tables') / INTERVAL
SAVE_MODELS = True
# First run notebooks/03_universe_master_table.ipynb to create master artifacts.

## 1. Load master and latest-bar inference tables

In [ ]:
master, latest, bundle, manifest = load_master(MASTER_DIRECTORY)
print('Master labeled rows:', f'{len(master):,}', 'stocks:', master.symbol.nunique())
print('Latest inference rows:', len(latest), 'stocks:', latest.symbol.nunique())
print('Training cutoff:', bundle['cutoff'], '| candle clusters:', manifest['cluster_k'])
print('Feature columns:', bundle['features'])
assert master['direction'].notna().all()
assert latest['direction'].isna().all()
display(master[['symbol','bar_timestamp','direction','next_return','next_adj_close']].head())

## 2. Apply saved preprocessing (no refitting)

In [ ]:
X, aligned, X_latest, aligned_latest = prepare_pooled_data(master, latest, bundle)
print('Training/evaluation feature matrix:', X.shape)
print('Latest-bar prediction matrix:', X_latest.shape)
print('Dropped incomplete historical rows:', len(master)-len(aligned))
print('Dropped incomplete latest rows:', len(latest)-len(aligned_latest))
assert X.columns.equals(X_latest.columns)
display(X.head())

## 3. Multiclass direction classification: 2 down, 0 no_change, 1 up

Compare a prior-probability baseline with histogram gradient boosting. ROC-AUC is **macro one-vs-rest** for three-class classification, when all classes occur in the evaluation period. Class probability columns are named by `model.classes_` (not assumed order).

In [ ]:
classification_models, classification_scores, (train_mask, test_mask) = classification_experiment(X, aligned, bundle['cutoff'])
display(classification_scores.sort_values('roc_auc', ascending=False, na_position='last'))
print('Train end:', aligned.loc[train_mask,'next_bar_timestamp'].max())
print('Test start:', aligned.loc[test_mask,'bar_timestamp'].min())
print('Class distribution in test:')
display(aligned.loc[test_mask,'direction'].value_counts().sort_index().to_frame('rows'))
classification_scores.set_index('model')[['accuracy','balanced_accuracy','roc_auc']].plot(kind='bar', rot=0, figsize=(9,4), title='Multiclass classification — chronological holdout')
plt.ylim(0,1); plt.tight_layout(); plt.show()

## 4. Next-bar return regression

For pooled stocks, predicting **next percentage return** rather than raw dollar price avoids forcing the regressor to learn radically different price scales. The notebook reconstructs predicted next adjusted close using the current adjusted close. This is a **one-step forecast**, not yet a multi-step trajectory.

In [ ]:
regression_models, regression_scores, (reg_train, reg_test) = regression_experiment(X, aligned, bundle['cutoff'])
display(regression_scores.sort_values('return_mae'))
regression_scores.set_index('model')[['return_mae','return_rmse']].plot(kind='bar', rot=0, figsize=(9,4), title='Next-bar return errors — lower is better')
plt.tight_layout(); plt.show()

## 5. Generate predictions from each stock’s latest completed bar

These rows have no known future label. We apply the **same saved transformer** and then the fitted models; do not compute accuracy on these predictions until the next candle is observed.

In [ ]:
class_predictions = predict_latest(classification_models, X_latest, aligned_latest, 'classification')
reg_predictions = predict_latest(regression_models, X_latest, aligned_latest, 'regression')
latest_predictions = class_predictions.merge(reg_predictions.drop(columns=['adj_close','interval_code']), on=['symbol','bar_timestamp'], validate='one_to_one')
print('Stocks with inference-ready rows:', len(latest_predictions))
display(latest_predictions.head(15))
# Example single ticker (if present):
display(latest_predictions.loc[latest_predictions.symbol.eq('AAPL')])

## 6. Save fitted models, metrics and inference outputs

Models are stored with a reference to the **master table transformer bundle**. Keep that bundle alongside deployed models; models expect its exact encoded feature ordering. Saved scores are exploratory, not production model approval.

In [ ]:
if SAVE_MODELS:
    class_dir = save_pooled_run(classification_models, classification_scores, class_predictions,
                                task='classification', master_directory=MASTER_DIRECTORY)
    reg_dir = save_pooled_run(regression_models, regression_scores, reg_predictions,
                              task='regression', master_directory=MASTER_DIRECTORY)
    print('Classification run:', class_dir)
    print('Regression run:', reg_dir)
else:
    print('Saving disabled')

## 7. Before multi-step forecasting and deployment

1. Add **walk-forward splits** and a final untouched holdout; fit target quantiles, K-means, scalers, encoders and estimators separately in each fold. Current master preprocessing is fixed at one training cutoff.
2. For a **15-bar regression path**, create `t+1..t+15` targets per stock, ensure each target timestamp is within its training fold, and forecast returns (or relative prices) rather than mixing raw dollar price scales. Do not generate future dates using naive calendar increments; use exchange sessions and interval schedules.
3. When promoting a model to Streamlit, store its entire feature definition and preprocessing bundle, and confirm all latest candles are complete. Prediction outputs are not realized returns or trading recommendations.